# ML Zoomcamp 2026 — Homework 10 (Kubernetes and Model Serving)
**Nicolau Calado Jofilsan** · 2026 cohort (DataTalks.Club)

**EN.** The lead-scoring API from homework 5 (a FastAPI service around a frozen scikit-learn pipeline, checked by SHA-256)
is packaged as a container image and deployed to a local Kubernetes cluster created with `kind`. The 2026 setup fixes the
image tag, the port, the service selector, the resource requests and the autoscaler bounds in version-controlled YAML.
The cluster work itself runs in a terminal with Docker, `kind` and `kubectl`; a helper script records the evidence in
`hw10_results.json`, and this notebook reads that file and the three manifests to answer the questions.

**PT.** A API de lead scoring do homework 5 (um serviço FastAPI em volta de um pipeline do scikit-learn congelado e
conferido por SHA-256) é empacotada como imagem de contêiner e implantada num cluster Kubernetes local criado com `kind`. O
setup de 2026 fixa a tag da imagem, a porta, o seletor do serviço, as requisições de recursos e os limites do autoscaler em
YAML versionado. O trabalho no cluster roda em um terminal com Docker, `kind` e `kubectl`; um script auxiliar registra as
evidências em `hw10_results.json`, e este notebook lê esse arquivo e os três manifestos para responder às questões.

Topics / Conteúdo: container image, `kind`, Pod, Deployment, Service (`ClusterIP`), port-forward, readiness probe,
HorizontalPodAutoscaler, resource requests and limits.

## 0. Setup and evidence files / Preparação e arquivos de evidência

In [1]:
import json
import os
import platform
import urllib.request

import yaml  # pip install pyyaml

print("python:", platform.python_version(), "| PyYAML:", yaml.__version__)

RAW = ("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
       "main/cohorts/2026/homework/10-kubernetes/")
os.makedirs("10-kubernetes", exist_ok=True)
for name in ("deployment.yaml", "service.yaml", "hpa.yaml"):
    path = os.path.join("10-kubernetes", name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(RAW + name, path)

with open("hw10_results.json", encoding="utf-8-sig") as f:   # written by run_hw10_kubernetes.ps1 (Windows adds a BOM)
    results = json.load(f)
print("run status:", results.get("status"), "| error:", results.get("error"))
print("model checksum (pipeline.bin):", results["pipeline_sha256"][:16] + "...")

python: 3.13.17 | PyYAML: 6.0.3
run status: ok | error: None
model checksum (pipeline.bin): 1646bbdcd38d4f04...


**EN.** The checksum above is the one the homework-5 analysis used, so the container serves the very same model.
**PT.** O checksum acima é o mesmo usado na análise do homework 5, então o contêiner serve exatamente o mesmo modelo.

## Q1. Local container response / Resposta do contêiner local

**EN.** The image is built from `05-deployment/Dockerfile` (Python 3.11.15 and `uv` pinned by digest, dependencies from
`uv.lock`), run with `docker run -p 9696:9696`, and called with the same request as `q6_test.py`. **PT.** A imagem é
construída a partir de `05-deployment/Dockerfile` (Python 3.11.15 e `uv` fixados por digest, dependências do `uv.lock`),
executada com `docker run -p 9696:9696` e chamada com a mesma requisição do `q6_test.py`.

In [2]:
q1 = results["q1_response"]
print("container response / resposta do contêiner:", q1)
print("\nQ1 =", f"{q1['conversion_probability']:.3f}", f" (unrounded {q1['conversion_probability']})")

container response / resposta do contêiner: {'conversion_probability': 0.769799, 'conversion': True}

Q1 = 0.770  (unrounded 0.769799)


**Answer / Resposta Q1: 0.770** — the grader accepts ±0.005. The request is a
complete lead record (organic search, technology, employed, Europe, 4 courses, income 80,304, 7 interactions, lead score
0.74), and the probability is deterministic because the model file is frozen. / o corretor aceita ±0,005. A requisição é um
registro completo de lead (busca orgânica, tecnologia, empregado, Europa, 4 cursos, renda 80.304, 7 interações, lead score
0,74), e a probabilidade é determinística porque o arquivo do modelo é congelado.

## Q2. Environment check / Verificação do ambiente

**EN.** Recorded, not graded. **PT.** Registrado, sem nota.

In [3]:
print(results["kind_version"])
print(results["kubectl_client_version"])

kind version 0.29.0
Client Version: v1.34.1
Kustomize Version: v5.7.1


## Q3. Smallest deployable unit / Menor unidade implantável

**EN.** A **Pod** is the smallest unit Kubernetes creates and manages: one or more containers that share a network
address and storage. A Deployment manages Pods (through a ReplicaSet), a Service gives them a stable address, and a Node is
the machine they run on. **PT.** Um **Pod** é a menor unidade que o Kubernetes cria e gerencia: um ou mais contêineres que
compartilham endereço de rede e armazenamento. Um Deployment gerencia Pods (por meio de um ReplicaSet), um Service dá a
eles um endereço estável e um Node é a máquina onde rodam.

**Answer / Resposta Q3: `Pod`**

## Q4. Default service type / Tipo do serviço padrão

**EN.** The cluster output below was captured with `kubectl get services` right after creating the cluster.
**PT.** A saída do cluster abaixo foi capturada com `kubectl get services` logo após criar o cluster.

In [4]:
print(results["services_default"])
row = next(line.split() for line in results["services_default"].splitlines() if line.startswith("kubernetes"))
print("\nQ4 =", row[1])

NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE
kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP   4s

Q4 = ClusterIP


**Answer / Resposta Q4: `ClusterIP`** — the built-in `kubernetes` service is only reachable from inside the cluster; no
cloud provider is involved. / o serviço embutido `kubernetes` só é alcançável de dentro do cluster; nenhum provedor de nuvem
participa.

## Q5. Loading the image / Carregando a imagem

**EN.** Kind nodes are containers with their own image store, so they cannot see images in the host Docker daemon. The
command that copies a local image into the cluster is `kind load docker-image zoomcamp-model:2026-hw10 --name
mlzoomcamp-2026`; `imagePullPolicy: IfNotPresent` in the Deployment then makes the node use that copy instead of trying to
pull the tag from a registry (where it does not exist). **PT.** Os nós do kind são contêineres com seu próprio
armazenamento de imagens, então não enxergam as imagens do Docker do computador. O comando que copia uma imagem local para o
cluster é `kind load docker-image zoomcamp-model:2026-hw10 --name mlzoomcamp-2026`; o `imagePullPolicy: IfNotPresent` no
Deployment faz o nó usar essa cópia em vez de tentar baixar a tag de um registro (onde ela não existe).

**Answer / Resposta Q5: `kind load docker-image`**

## Q6. Container port / Porta do contêiner

In [5]:
with open("10-kubernetes/deployment.yaml", encoding="utf-8") as f:
    deployment = yaml.safe_load(f)
container = deployment["spec"]["template"]["spec"]["containers"][0]
print("image:", container["image"], "| pull policy:", container["imagePullPolicy"])
print("resources:", container["resources"])
print("readiness probe:", container["readinessProbe"])
print("\nQ6 =", container["ports"][0]["containerPort"])
print("pods after rollout / pods depois da implantação:\n" + results["pods"])

image: zoomcamp-model:2026-hw10 | pull policy: IfNotPresent
resources: {'requests': {'memory': '64Mi', 'cpu': '100m'}, 'limits': {'memory': '256Mi', 'cpu': '500m'}}
readiness probe: {'httpGet': {'path': '/health', 'port': 9696}, 'initialDelaySeconds': 2, 'periodSeconds': 5}

Q6 = 9696
pods after rollout / pods depois da implantação:
NAME                            READY   STATUS    RESTARTS   AGE
subscription-7d87ff6fb4-9fn55   1/1     Running   0          7s


**Answer / Resposta Q6: `9696`** — the same port the Uvicorn server listens on in the Dockerfile. The readiness probe calls
`/health` on that port, and a Pod only receives traffic once it answers. / a mesma porta em que o servidor Uvicorn escuta no
Dockerfile. A sonda de prontidão chama `/health` nessa porta, e um Pod só recebe tráfego depois que responde.

## Q7. Service selector / Seletor do serviço

In [6]:
with open("10-kubernetes/service.yaml", encoding="utf-8") as f:
    service = yaml.safe_load(f)
selector = service["spec"]["selector"]
print("type:", service["spec"]["type"], "| ports:", service["spec"]["ports"])
print("Pod labels in the Deployment / rótulos dos Pods:", deployment["spec"]["template"]["metadata"]["labels"])
print("\nQ7 =", ": ".join(next(iter(selector.items()))))
assert selector == deployment["spec"]["template"]["metadata"]["labels"], "selector does not match the Pod labels"

q7 = results["q7_response"]
diff = abs(q7["conversion_probability"] - q1["conversion_probability"])
print("\nthrough the port-forward / pelo port-forward:", q7)
print(f"difference from Q1 / diferença em relação à Q1: {diff:.6f}  (limit 0.005)")
assert diff <= 0.005

type: ClusterIP | ports: [{'port': 80, 'targetPort': 9696}]
Pod labels in the Deployment / rótulos dos Pods: {'app': 'subscription'}

Q7 = app: subscription

through the port-forward / pelo port-forward: {'conversion_probability': 0.769799, 'conversion': True}
difference from Q1 / diferença em relação à Q1: 0.000000  (limit 0.005)


**Answer / Resposta Q7: `app: subscription`** — the Service sends traffic to the Pods whose label matches the selector, and the
Deployment puts exactly that label on its Pods. Without a match the Service would have no endpoints. The Service uses
`ClusterIP` on port 80 forwarding to 9696, and `kubectl port-forward service/subscription 9696:80` opens it on the local
machine; the probability is the same as in Q1. / o Service envia o tráfego aos Pods cujo rótulo bate com o seletor, e o
Deployment coloca exatamente esse rótulo nos seus Pods. Sem correspondência o Service ficaria sem destinos. O Service usa
`ClusterIP` na porta 80 encaminhando para 9696, e `kubectl port-forward service/subscription 9696:80` o abre na máquina
local; a probabilidade é a mesma da Q1.

## Q8. HPA configuration / Configuração do HPA

In [7]:
with open("10-kubernetes/hpa.yaml", encoding="utf-8") as f:
    hpa = yaml.safe_load(f)
spec = hpa["spec"]
target = spec["metrics"][0]["resource"]["target"]["averageUtilization"]
print("apiVersion:", hpa["apiVersion"], "| min:", spec["minReplicas"], "| max:", spec["maxReplicas"],
      "| CPU target:", f"{target}%")
print("\nQ8 =", spec["maxReplicas"])
print("\nkubectl get hpa:\n" + results["hpa"])

apiVersion: autoscaling/v2 | min: 1 | max: 3 | CPU target: 20%

Q8 = 3

kubectl get hpa:
NAME               REFERENCE                 TARGETS              MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   cpu: <unknown>/20%   1         3         0          5s


**Answer / Resposta Q8: `3`** — `maxReplicas: 3`. The target is 20% average CPU **of the request** (100m), so scaling starts
at about 20 millicores per Pod, a low threshold chosen to make the autoscaler easy to trigger. A CPU value shown as
`<unknown>` means the cluster has no metrics-server yet; it is an infrastructure status, not an answer. / `maxReplicas: 3`.
A meta é 20% de CPU média **em relação à requisição** (100m), então o escalonamento começa perto de 20 milicores por Pod, um
limiar baixo escolhido para facilitar o disparo do autoscaler. Um valor de CPU `<unknown>` significa que o cluster ainda não
tem metrics-server; é um estado da infraestrutura, não uma resposta.

### Homework 10 — answers / respostas

| Question | Answer |
|---|---|
| Q1 `conversion_probability` from the container (3 decimals) | **0.770** |
| Q2 Versions of `kind` and `kubectl` | recorded, not graded |
| Q3 Smallest deployable unit | **`Pod`** |
| Q4 Type of the `kubernetes` service | **`ClusterIP`** |
| Q5 Command that loads the image | **`kind load docker-image`** |
| Q6 Container port | **`9696`** |
| Q7 Selector value | **`app: subscription`** |
| Q8 `maxReplicas` | **`3`** |

### Notes / Observações

**EN.** 1) *Everything is pinned.* The model file is checked by SHA-256, the base image and `uv` are pinned by digest and the
dependencies by lockfile, so the probability does not depend on when or where the image is built. 2) *Why `ClusterIP` plus
port-forward.* A `LoadBalancer` service needs a cloud (or an extra component such as MetalLB) to get an address, so the
homework reaches the service through `kubectl port-forward`, which listens only on the local machine. 3) *Requests and the
autoscaler.* CPU utilization in the HPA is measured against the Pod's CPU **request**, not its limit or the machine's CPU;
changing the request changes when scaling happens. 4) *Not an access-controlled service.* The API has no authentication;
the port-forward is local, but exposing it through a real `LoadBalancer` or an ingress would need authentication and
rate limiting first. 5) *Scope.* One replica on one local node tests the manifests, not availability, rolling updates under
load or cluster failures.

**PT.** 1) *Tudo está fixado.* O arquivo do modelo é conferido por SHA-256, a imagem base e o `uv` são fixados por digest e as
dependências por lockfile, então a probabilidade não depende de quando nem de onde a imagem é construída. 2) *Por que
`ClusterIP` com port-forward.* Um serviço `LoadBalancer` precisa de uma nuvem (ou de um componente extra como o MetalLB) para
receber um endereço, então o enunciado acessa o serviço por `kubectl port-forward`, que escuta só na máquina local. 3)
*Requisições e o autoscaler.* A utilização de CPU no HPA é medida em relação à **requisição** de CPU do Pod, não ao limite
nem à CPU da máquina; mudar a requisição muda quando o escalonamento acontece. 4) *Não é um serviço com controle de
acesso.* A API não tem autenticação; o port-forward é local, mas expô-la por um `LoadBalancer` real ou um ingress exigiria
autenticação e limite de requisições antes. 5) *Escopo.* Uma réplica em um nó local testa os manifestos, não disponibilidade,
atualizações graduais sob carga nem falhas do cluster.